In [1]:
import cv2 # compter vision
import mediapipe as mp # responsable de la detection des doigts et main 
import tensorflow as tf
from matplotlib import pyplot as plt
import csv
import numpy as np

In [2]:
mp_mains = mp.solutions.hands
mp_dessin = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
# à travers cela on crée des raccourcis vers des modules de mediapipe

In [3]:
Mains = mp_mains.Hands(model_complexity=0,min_detection_confidence=0.7,min_tracking_confidence=0.7)
# on initialise un objet Mains qu'on va utiliser plus tard. Les paramètres sont ceux du modele de deep learning de la lib mediapipe

In [4]:
cap = cv2.VideoCapture(0) 
#cap est un objet capable de lire l'output de la webcam numéro 0

In [5]:
ret, image = cap.read() # cap.read() renvoie deux variables ret : True ou False et image qui comporte l'image

In [6]:
resultats = Mains.process(image) # the magic happens here : c'est la que le modèle travaille et effectue la prediction

In [7]:
# Première chose à faire est de collecter notre data : la data c'est toutes les gestes qu'on veut reconnaitre

In [8]:
def normaliser_landmarks(points,wrist,signe):
    L = []
    for couples in points:
        x = couples[0] - wrist[0]
        y = couples[1] - wrist[1]
        L.append(x)
        L.append(y)
    return [L,signe]    

In [9]:
fichier_csv = open('dataset.csv','a')
writer = csv.writer(fichier_csv)
def remplir_csv(liste_normalisée,signe): 
    writer.writerow(liste_normalisée+[signe])

In [10]:
#maintenant on peut commencer notre boucle : 
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap = cv2.VideoCapture(0) 
while cap.isOpened():
    ret, image = cap.read()
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # OpenCV lit du BGR mais MediaPipe attend du RGB
    results = Mains.process(image)
    if results.multi_hand_landmarks: #si le résultat n'est pas null
        for hand_landmarks in results.multi_hand_landmarks: # on parcourt les éléments de la liste results.multi_hand_landmarks
            mp_dessin.draw_landmarks(image,hand_landmarks,mp_mains.HAND_CONNECTIONS,mp_drawing_styles.get_default_hand_landmarks_style(),mp_drawing_styles.get_default_hand_connections_style())
            # on affiche des points sur nos doigts grace à draw_landmarks
            
            points = []
            for lm in hand_landmarks.landmark:
                    h, w, _ = image.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    points.append((cx, cy)) 
        wrist = points[0]       
        # quand on clique sur 'p' on va enregistrer le frame qu'on a capturer (on enregistre directement les landmarks) : P pour peace sign
        key = cv2.waitKey(5) & 0xFF
        if key == ord('p'):
            res = normaliser_landmarks(points,wrist,"peace")
            print(res[0],res[1])
            remplir_csv(res[0],res[1])
  
            
        # quand on clique sur 't' on va enregistrer le frame qu'on a capturer (on enregistre directement les landmarks) : t pour thumbs up
        
        if key == ord('t'):
            res = normaliser_landmarks(points,wrist,"thumb")
            print(res[0],res[1])
            remplir_csv(res[0],res[1])
        
        # quand on clique sur 'g' on va enregistrer le frame qu'on a capturer (on enregistre directement les landmarks) : g pour gojo_domain
        
        if key == ord('g'):
            res = normaliser_landmarks(points,wrist,"gojo_domain")
            print(res[0],res[1])
            remplir_csv(res[0],res[1])

        # quand on clique sur 'k' on va enregistrer le frame qu'on a capturer (on enregistre directement les landmarks) : k pour kon
        
        if key == ord('k'):
            res = normaliser_landmarks(points,wrist,"kon")
            print(res[0],res[1])
            remplir_csv(res[0],res[1])     

        # quand on clique sur 'o' on va enregistrer le frame qu'on a capturer (on enregistre directement les landmarks) : o pour OK
        
        if key == ord('o'):
            res = normaliser_landmarks(points,wrist,"OK")
            print(res[0],res[1])
            remplir_csv(res[0],res[1])

        # quand on clique sur 'L' on va enregistrer le frame qu'on a capturer (on enregistre directement les landmarks) : L pour Loser
        
        if key == ord('l'):
            res = normaliser_landmarks(points,wrist,"Loser")
            print(res[0],res[1])
            remplir_csv(res[0],res[1])
            
            
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    cv2.imshow("Hand Tracking", image)       
    # Pour pouvoir sortir de la boucle        
    if cv2.waitKey(5) & 0XFF == ord('q'):
                break     
        
    
    
        
        
    
    
cap.release() # pour fermer la cam
cv2.destroyAllWindows()

 

In [11]:
# On a pris des photos de nos différents signes et on les a stocké dans un fichier csv : On passe à la partie suivante

In [12]:
import pandas as pd

In [13]:
data = pd.read_csv("dataset.csv") # import our data

In [14]:
data

,x0,y0,x1,y1,x2,y2,x3,y3,x4,y4,...,y16,x17,y17,x18,y18,x19,y19,x20,y20,sign
0,0,0,39,-6,60,-29,36,-48,4,-63,...,-176,-44,-98,-40,-90,-25,-62,-10,-41,peace
1,0,0,41,-7,64,-31,39,-52,8,-68,...,-97,-44,-105,-39,-92,-23,-58,-6,-34,peace
2,0,0,41,-8,62,-35,32,-58,-1,-76,...,-208,-46,-100,-40,-95,-25,-68,-12,-46,peace
3,0,0,39,-10,57,-37,26,-57,-6,-71,...,-82,-52,-108,-45,-91,-29,-58,-11,-35,peace
4,0,0,37,-11,53,-41,20,-61,-13,-76,...,-58,-53,-105,-49,-91,-33,-60,-15,-39,peace
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2452,0,0,36,-36,50,-70,73,-92,101,-111,...,-187,9,-89,18,-134,22,-164,26,-190,kon
2453,0,0,34,-36,50,-71,76,-94,103,-114,...,-189,11,-89,20,-135,24,-165,28,-190,kon
2454,0,0,32,-37,49,-72,76,-95,104,-113,...,-197,11,-85,20,-132,22,-162,24,-189,kon
2455,0,0,33,-40,48,-73,74,-97,102,-114,...,-199,10,-86,20,-132,23,-163,26,-190,kon


In [15]:
print(data.shape)

(2457, 43)


In [16]:
data.info()
# on remarque que notre colonne sign est composée d'objets non pas de reel ou bool donc il va falloir réglèr ça en créeant des paramètres dummy

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2457 entries, 0 to 2456
Data columns (total 43 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   x0      2457 non-null   int64 
 1   y0      2457 non-null   int64 
 2   x1      2457 non-null   int64 
 3   y1      2457 non-null   int64 
 4   x2      2457 non-null   int64 
 5   y2      2457 non-null   int64 
 6   x3      2457 non-null   int64 
 7   y3      2457 non-null   int64 
 8   x4      2457 non-null   int64 
 9   y4      2457 non-null   int64 
 10  x5      2457 non-null   int64 
 11  y5      2457 non-null   int64 
 12  x6      2457 non-null   int64 
 13  y6      2457 non-null   int64 
 14  x7      2457 non-null   int64 
 15  y7      2457 non-null   int64 
 16  x8      2457 non-null   int64 
 17  y8      2457 non-null   int64 
 18  x9      2457 non-null   int64 
 19  y9      2457 non-null   int64 
 20  x10     2457 non-null   int64 
 21  y10     2457 non-null   int64 
 22  x11     2457 non-null   

In [17]:
data = pd.get_dummies(data.sign)
data

,Loser,OK,gojo_domain,kon,peace,thumb
0,False,False,False,False,True,False
1,False,False,False,False,True,False
2,False,False,False,False,True,False
3,False,False,False,False,True,False
4,False,False,False,False,True,False
...,...,...,...,...,...,...
2452,False,False,False,True,False,False
2453,False,False,False,True,False,False
2454,False,False,False,True,False,False
2455,False,False,False,True,False,False


In [18]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2457 entries, 0 to 2456
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Loser        2457 non-null   bool 
 1   OK           2457 non-null   bool 
 2   gojo_domain  2457 non-null   bool 
 3   kon          2457 non-null   bool 
 4   peace        2457 non-null   bool 
 5   thumb        2457 non-null   bool 
dtypes: bool(6)
memory usage: 14.5 KB


In [19]:
# apres les recherches il est optimale de garder une seule colonne sign et attribuer a chque sign un chiffre
data = pd.read_csv("dataset.csv") # import our data

data["sign"] = data["sign"].map({
    "peace": 0,
    "thumb": 1,
    "gojo_domain": 2,
    "Loser": 3,
    "OK": 4,
    "kon": 5,
})
data

,x0,y0,x1,y1,x2,y2,x3,y3,x4,y4,...,y16,x17,y17,x18,y18,x19,y19,x20,y20,sign
0,0,0,39,-6,60,-29,36,-48,4,-63,...,-176,-44,-98,-40,-90,-25,-62,-10,-41,0
1,0,0,41,-7,64,-31,39,-52,8,-68,...,-97,-44,-105,-39,-92,-23,-58,-6,-34,0
2,0,0,41,-8,62,-35,32,-58,-1,-76,...,-208,-46,-100,-40,-95,-25,-68,-12,-46,0
3,0,0,39,-10,57,-37,26,-57,-6,-71,...,-82,-52,-108,-45,-91,-29,-58,-11,-35,0
4,0,0,37,-11,53,-41,20,-61,-13,-76,...,-58,-53,-105,-49,-91,-33,-60,-15,-39,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2452,0,0,36,-36,50,-70,73,-92,101,-111,...,-187,9,-89,18,-134,22,-164,26,-190,5
2453,0,0,34,-36,50,-71,76,-94,103,-114,...,-189,11,-89,20,-135,24,-165,28,-190,5
2454,0,0,32,-37,49,-72,76,-95,104,-113,...,-197,11,-85,20,-132,22,-162,24,-189,5
2455,0,0,33,-40,48,-73,74,-97,102,-114,...,-199,10,-86,20,-132,23,-163,26,-190,5


In [20]:
# Séparer nos variables Features et Target : 

X = data.drop("sign",axis=1)
Y = data["sign"]

# creer nos données de training et de test 80% et 20% :

from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size = 0.2)


In [21]:
from sklearn.linear_model import LogisticRegression # LogisticRegression au lieu de LinearRegression parce que ceci est un probleme de classification non pas de régression linéaire

model = LogisticRegression(max_iter=2000,class_weight="balanced")
model.fit(X_train, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [22]:
model.score(X_test,Y_test)

0.9979674796747967

In [23]:
# on peut utiliser randomforest qui simule plusieurs arbres de décision qui peut améliorer notre précision

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train, Y_train)

print(model.score(X_test, Y_test))

0.9979674796747967


In [24]:
print(data["sign"].value_counts())

sign
4    477
5    459
1    456
0    400
3    362
2    303
Name: count, dtype: int64


In [25]:
model.score(X_test,Y_test)
# on regarde la précision de notre modèle qui est très impréssionnante

0.9979674796747967

In [26]:
# on définit la fonction suivante qui sert a écrire une ligne dans notre fichier de texte qu'on utilise dans notre boucle
# la ligne est bien le landmark de notre signe input qu'on veut classifier
def remplir_csv(liste_normalisée,signe): 
    fichier_csv = open('test.csv','w')
    writer = csv.writer(fichier_csv)
    writer.writerow(["x0","y0","x1","y1","x2","y2","x3","y3","x4","y4","x5","y5","x6","y6","x7","y7","x8","y8","x9","y9","x10","y10","x11","y11","x12","y12","x13","y13","x14","y14","x15","y15","x16","y16","x17","y17","x18","y18","x19","y19","x20","y20"]+["sign"])
    writer.writerow(liste_normalisée+[signe])

In [27]:
# une fonction qui renvoie une liste de proba d'un resultat par ex : [0.1,0.2,0.5,0.1,0.1] cela nous donne la proba de chaque signe selon notre modele
def donner_proba(fichier_test):
    data_test = pd.read_csv(fichier_test)
    X = data_test.drop("sign",axis=1)
    res = model.predict_proba(X)
    return res

In [28]:
# on veut que notre prédicition soit précise c pour ça qu'on cherche la proba la plus elevée et on la compare a notre threshold qui est 0.65 en bas
def limite_confiance(liste_proba):
    max_val = np.max(liste_proba)
    indice = np.argmax(liste_proba)
    return max_val, indice

In [29]:
# cette fonction fait appel au module predict de notre modèle qui nous donne le résultat
def predire(fichier_test):
    data_test = pd.read_csv(fichier_test)
    X = data_test.drop("sign",axis=1)
    res = model.predict(X)
    return res

In [30]:
# le programme nous renvoie 0 ou 1 ou 2 ... parce que c défini comme ça. Donc ici on fait le décodage du résultat.

def afficher_signe(frame,proba,indice): # fonction qui prend en paramètre le résultat de la prédiction et renvoie le nom de la prediction sur l'écran
   if (proba>0.65):
        match indice:
            case 0:
                text = "Peace"
            case 1:
                text = "thumb"
            case 2:
                text = "gojo_domain"
            case 3:
                text = "Loser"
            case 4:
                text = "OK"
            case 5:
                text = "kon"
            
        cv2.putText(frame,text,(50,50),cv2.FONT_HERSHEY_SIMPLEX,1,(0, 255, 0),2,cv2.LINE_AA)
   else:
        cv2.putText(frame,"unknown",(50,50),cv2.FONT_HERSHEY_SIMPLEX,1,(0, 255, 0),2,cv2.LINE_AA)

In [31]:
#maintenant on peut commencer notre boucle : 
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap = cv2.VideoCapture(0) 
while cap.isOpened():
    ret, image = cap.read()
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # OpenCV lit du BGR mais MediaPipe attend du RGB
    results = Mains.process(image)
    if results.multi_hand_landmarks: #si le résultat n'est pas null
        for hand_landmarks in results.multi_hand_landmarks: # on parcourt les éléments de la liste results.multi_hand_landmarks
            #mp_dessin.draw_landmarks(image,hand_landmarks,mp_mains.HAND_CONNECTIONS,mp_drawing_styles.get_default_hand_landmarks_style(),mp_drawing_styles.get_default_hand_connections_style())            
            points = []
            for lm in hand_landmarks.landmark:
                    h, w, _ = image.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    points.append((cx, cy)) 
        wrist = points[0]      

        # the magic happens here ; on appelle toutes nos fonctions
        res = normaliser_landmarks(points,wrist,"???")
        remplir_csv(res[0],res[1])
        answer = predire("test.csv")
        res_proba = donner_proba("test.csv")
        proba,indice = limite_confiance(res_proba)
        afficher_signe(image,proba,indice)    
            
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    cv2.imshow("Hand Tracking", image)       
    # Pour pouvoir sortir de la boucle        
    if cv2.waitKey(5) & 0XFF == ord('q'):
                break     
        
    
    
        
        
    
    
cap.release() # pour fermer la cam
cv2.destroyAllWindows()
